# Download data

In [142]:
import pandas as pd
from sklearn.covariance import LedoitWolf
import numpy as np

In [143]:
import refinitiv.data as rd
rd.open_session()

<refinitiv.data.session.Definition object at 0x17743d910 {name='workspace'}>

# In-sample

In [144]:
djia = rd.get_data(
    universe=".DJI",
    fields=[
        "TR.IndexConstituentRIC",
        "TR.IndexConstituentName"
    ],
    parameters={"SDate": "2023-12-31"}
)


In [145]:
rics = djia["Constituent RIC"].tolist()

rics = [ric for ric in rics if ric != "DOW.N"]


mktcap = rd.get_data(
    universe=rics,
    fields=[
        "TR.CompanyMarketCap",
        "TR.FloatSharesOutstanding",
        "TR.PriceClose"
    ],
    parameters={"SDate": "2023-12-31"}
)
mktcap["BenchWeight"] = (
    mktcap["Company Market Cap"] /
    mktcap["Company Market Cap"].sum()
)


In [146]:
prices_10y = rd.get_data(
    universe=rics,
    fields=["TR.PriceClose", "TR.PriceClose.Date"],
    parameters={
        "SDate": "2013-12-31",
        "EDate": "2023-12-31",
        "Frq": "D"
    }
)


In [147]:
prices_10y.loc[
    (prices_10y["Instrument"] == "DOW.N") &
    prices_10y["Price Close"].notna(),
    "Date"
].min()


NaT

DJIA official size: 30

After exclusion: 29 stocks

In [148]:
prices_10y = prices_10y.loc[prices_10y['Instrument'] != 'DOW.N']

In [149]:
dups = prices_10y[
    prices_10y.duplicated(
        subset=["Date", "Instrument"],
        keep=False
    )
].sort_values(["Instrument", "Date"])

dups.head(20)

,Instrument,Price Close,Date


In [150]:
import numpy as np
returns_10y = prices_10y.pivot(index="Date", columns="Instrument", values="Price Close")
returns_10y = np.log(returns_10y / returns_10y.shift(1)).dropna()


In [151]:
returns_10y.isna().any().any()


False

In [152]:
end_date = pd.Timestamp("2023-12-31")
start_5y = end_date - pd.DateOffset(years=5)

returns_5y = returns_10y.loc[start_5y:end_date]


In [153]:
returns_5y.index

DatetimeIndex(['2018-12-31', '2019-01-02', '2019-01-03', '2019-01-04',
               '2019-01-07', '2019-01-08', '2019-01-09', '2019-01-10',
               '2019-01-11', '2019-01-14',
               ...
               '2023-12-15', '2023-12-18', '2023-12-19', '2023-12-20',
               '2023-12-21', '2023-12-22', '2023-12-26', '2023-12-27',
               '2023-12-28', '2023-12-29'],
              dtype='datetime64[ns]', name='Date', length=1259, freq=None)

In [154]:
returns_5y.isna().sum().sum()



0

In [ ]:
esg = rd.get_data(
    universe=rics,
    fields=[
        "TR.TRESGScore.date",
        "TR.TRESGScore",
    ],
    parameters={
        "Period": "FY0",   # most recent fiscal year
        "Frq": "FY",
        "SDate": "0",
        "EDate": "-1"
    }
)


In [156]:
esg

,Instrument,Date,ESG Score
0,GS.N,2023-12-31,83.848062
1,GS.N,2022-12-31,86.705328
2,NKE.N,2023-05-31,70.082066
3,NKE.N,2022-05-31,73.285228
4,CSCO.OQ,2023-07-29,82.498168
5,CSCO.OQ,2022-07-30,82.514291
6,JPM.N,2023-12-31,79.922723
7,JPM.N,2022-12-31,78.981666
8,DIS.N,2023-09-30,67.681608
9,DIS.N,2022-10-01,69.052736


In [127]:
esg["Date"].dt.year.value_counts()



Date
2023    30
2024    15
2022    13
Name: count, dtype: int64

In [128]:
esg["Date"] = pd.to_datetime(esg["Date"], errors="coerce")

esg = (
    esg[esg["Date"].dt.year == 2023]   # keep any date in 2023
    .sort_values("Date")
    .groupby("Instrument", as_index=False)
    .last()
)



In [129]:
esg.isna().sum()


Instrument    0
Date          0
ESG Score     0
dtype: int64

In [130]:
len(esg)

29

In [131]:
returns_10y.to_parquet("data/eturns_10y.parquet")
returns_5y.to_parquet("data/returns_5y.parquet")

esg.to_csv("data/esg.csv")
universe = pd.DataFrame({
    "Instrument": returns_10y.columns
})

universe.to_csv("data/universe_djia.csv", index=False)
mktcap.to_csv("data/benchmark_weights.csv")



# Out-of-Sample

In [132]:
prices_oos = rd.get_data(
    universe=rics,
    fields=["TR.PriceClose", "TR.PriceClose.Date"],
    parameters={
        "SDate": "2024-01-01",
        "EDate": "2024-12-31",
        "Frq": "D"
    }
)

In [133]:
prices_oos = (
    prices_oos
    .dropna(subset=["Price Close"])
    .sort_values(["Instrument", "Date"])
    .drop_duplicates(subset=["Instrument", "Date"], keep="last")
)

In [134]:
prices_oos

,Instrument,Price Close,Date
5544,AAPL.OQ,185.64,2024-01-02
5545,AAPL.OQ,184.25,2024-01-03
5546,AAPL.OQ,181.91,2024-01-04
5547,AAPL.OQ,181.18,2024-01-05
5548,AAPL.OQ,185.56,2024-01-08
...,...,...,...
6043,WMT.OQ,92.68,2024-12-24
6044,WMT.OQ,92.79,2024-12-26
6045,WMT.OQ,91.66,2024-12-27
6046,WMT.OQ,90.57,2024-12-30


In [135]:
prices_oos_wide = prices_oos.pivot(
    index="Date",
    columns="Instrument",
    values="Price Close"
)
prices_oos_wide

Instrument,AAPL.OQ,AMGN.OQ,AXP.N,BA.N,CAT.N,CRM.N,CSCO.OQ,CVX.N,DIS.N,GS.N,...,MRK.N,MSFT.OQ,NKE.N,PG.N,TRV.N,UNH.N,V.N,VZ.N,WBA.OQ^H25,WMT.OQ
Date,,,,,,,,,,,,,,,,,,,,,
2024-01-02,185.64,297.39,188.31,251.76,292.71,256.13,50.51,149.48,90.71,388.3,...,113.24,370.87,106.55,148.74,191.42,539.34,258.87,38.88,26.65,53.096614
2024-01-03,184.25,300.69,186.32,243.91,284.3,251.84,50.51,152.33,91.65,381.79,...,114.77,370.6,104.04,147.84,191.3,542.03,257.98,39.16,25.57,53.099947
2024-01-04,181.91,303.17,187.14,244.94,286.1,251.24,50.08,150.66,90.56,382.95,...,117.01,367.94,102.3,148.65,192.54,545.42,259.61,39.37,24.26,52.586614
2024-01-05,181.18,303.0,189.06,249.0,288.93,251.12,50.09,150.4,90.9,386.44,...,117.22,367.75,102.08,147.42,193.07,537.38,259.69,40.2,25.01,52.236614
2024-01-08,185.56,310.88,189.21,229.0,292.25,260.87,50.46,149.5,91.55,388.86,...,117.38,374.69,103.62,148.69,192.31,536.52,262.54,40.1,25.63,52.749947
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-24,258.2,264.49,303.46,179.34,367.57,344.43,59.85,143.84,112.56,582.79,...,99.45,439.33,76.79,168.94,242.88,506.1,320.65,39.8,9.19,92.68
2024-12-26,259.02,263.18,303.99,180.38,367.12,341.72,59.98,143.98,112.55,581.23,...,99.87,438.11,76.94,170.16,243.73,511.15,320.91,39.96,9.68,92.79
2024-12-27,255.59,262.65,301.05,180.72,364.86,338.45,59.61,144.0,111.55,576.18,...,99.7,430.53,76.42,169.53,241.41,509.99,318.66,39.92,9.62,91.66


In [136]:
returns_oos = np.log(prices_oos_wide / prices_oos_wide.shift(1)).dropna()


In [137]:
returns_oos = returns_oos[returns_10y.columns]

In [138]:
returns_oos.shape

(251, 29)

In [139]:
returns_oos.isna().sum().sum()

0

In [140]:
returns_oos.to_parquet("data/returns_oos_2024.parquet")

In [141]:
rd.close_session()